In [1]:
pip install pygame

#Imports and Configuration

In [2]:
# ===================================
# Import all required libraries and define simulation constants

import pygame
import random
import pickle
import os
import sys

# ================================
# Constants for Simulation
# ================================
FPS = 120  # Very high FPS for faster simulation
SCREEN_WIDTH = 400
SCREEN_HEIGHT = 600

FLOORS = 6  # Basement + 5 floors above
FLOOR_HEIGHT = SCREEN_HEIGHT // FLOORS
ELEVATOR_WIDTH = 80
ELEVATOR_HEIGHT = FLOOR_HEIGHT - 10
ELEVATOR_SPEED = 30  # pixels per frame, very fast movement for speed

# ================================
# Reinforcement Learning Parameters
# ================================
ALPHA = 0.1      # Learning rate
GAMMA = 0.95     # Discount factor
EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY = 0.995  # decay per episode

# ================================
# Action Definitions
# ================================
ACTION_UP = 0
ACTION_DOWN = 1
ACTION_STAY = 2
ACTIONS = [ACTION_UP, ACTION_DOWN, ACTION_STAY]

# ================================
# UI Colors
# ================================
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GRAY = (230, 230, 230)
RED = (220, 50, 50)
GREEN = (50, 180, 50)
BLUE = (50, 50, 200)
YELLOW = (240, 230, 140)

# ================================
# Floor Names (for UI Display)
# ================================
FLOOR_NAMES = ['B', '1', '2', '3', '4', '5']

# ================================
# Model Save Path
# ================================
MODEL_SAVE_PATH = "best_q_table.pkl"

print("✓ Libraries imported and configuration set")

pygame 2.6.1 (SDL 2.28.4, Python 3.11.12)
Hello from the pygame community. https://www.pygame.org/contribute.html
✓ Libraries imported and configuration set


#Environment and Agent Classes

In [3]:
# ====================================
# Define the elevator environment and Q-learning agent classes

class ElevatorEnv:
    """
    Elevator environment modeling realistic pickup and drop requests.

    - Passengers request pickup at origin floors, with a destination floor.
    - Elevator serves requests by picking up passengers and dropping them off.
    - State: current floor, pickup requests vector per floor, drop requests vector per floor.
    - Actions: move up, down, stay(open door).
    - Reward: penalize waiting time (-1 per step per waiting passenger),
      reward +30 for each successful dropoff, +15 for each pickup.
    """

    def __init__(self):
        self.num_floors = FLOORS
        self.current_floor = 0  # Start at basement = floor 0

        # Separate lists for pickup requests and passengers inside elevator (drop requests)
        # pickup_requests[floor] = list of destination floors requested from that floor
        self.pickup_requests = [[] for _ in range(self.num_floors)]
        # passengers are destination floors inside elevator
        self.passengers = []

        self.time_step = 0
        self.max_time_steps = 300

    def reset(self):
        self.current_floor = 0
        self.pickup_requests = [[] for _ in range(self.num_floors)]
        self.passengers = []
        self.time_step = 0

        # Initial random requests for simulation start
        for _ in range(5):
            origin = random.randint(0, self.num_floors - 1)
            dest = random.randint(0, self.num_floors - 1)
            while dest == origin:
                dest = random.randint(0, self.num_floors - 1)
            self.pickup_requests[origin].append(dest)

        return self._get_state()

    def _get_state(self):
        """
        State represented as:
          - current floor (integer)
          - pickup requests boolean vector (floor has pickup requests or not)
          - drop requests boolean vector (passenger destinations floors or not)
        """

        pickup_vec = tuple(int(len(self.pickup_requests[floor]) > 0) for floor in range(self.num_floors))
        drop_vec = [0]*self.num_floors
        for dest_floor in self.passengers:
            drop_vec[dest_floor] = 1
        drop_vec = tuple(drop_vec)

        return (self.current_floor, pickup_vec, drop_vec)

    def step(self, action):
        """
        Takes action and updates environment state.
        Returns: new_state, reward, done, info
        """

        self.time_step += 1

        # Calculate negative reward for waiting passengers:
        waiting_passengers_count = sum(len(dest_list) for dest_list in self.pickup_requests)
        negative_reward = -1 * (waiting_passengers_count)  # penalize waiting passengers per step

        reward = negative_reward
        done = False

        # New passengers arrive probabilistically (0.1 chance per floor per step)
        self._generate_new_passengers()

        # Move elevator or stay
        if action == ACTION_UP and self.current_floor < self.num_floors - 1:
            self.current_floor += 1
        elif action == ACTION_DOWN and self.current_floor > 0:
            self.current_floor -= 1
        elif action == ACTION_STAY:
            # Elevator door opens:
            # 1) drop off passengers whose destination is current floor
            drop_count = self.passengers.count(self.current_floor)
            reward += drop_count * 30
            self.passengers = [d for d in self.passengers if d != self.current_floor]

            # 2) pick up all passengers waiting at this floor
            pickup_count = len(self.pickup_requests[self.current_floor])
            reward += pickup_count * 15

            # add picked passengers to elevator's passengers list (drop requests)
            self.passengers.extend(self.pickup_requests[self.current_floor])
            self.pickup_requests[self.current_floor] = []

        # Done if max time steps reached
        if self.time_step >= self.max_time_steps:
            done = True

        new_state = self._get_state()
        return new_state, reward, done, {}

    def _generate_new_passengers(self):
        """
        Randomly generate new passengers requesting elevator.
        Each passenger has origin floor and destination floor.
        """

        prob_new_passenger = 0.1
        for floor in range(self.num_floors):
            if random.random() < prob_new_passenger:
                origin = floor
                # Destination different from origin
                dest = random.randint(0, self.num_floors - 1)
                while dest == origin:
                    dest = random.randint(0, self.num_floors - 1)
                self.pickup_requests[origin].append(dest)

    def pending_pickups_count(self):
        return sum(len(dest_list) for dest_list in self.pickup_requests)

    def passengers_count(self):
        return len(self.passengers)


class QLearningAgent:
    """
    Q-Learning agent for Elevator environment.
    Uses a dictionary to map (state, action) to Q-values.
    """

    def __init__(self, num_floors):
        self.alpha = ALPHA
        self.gamma = GAMMA
        self.epsilon = EPSILON_START
        self.epsilon_min = EPSILON_END
        self.epsilon_decay = EPSILON_DECAY
        self.num_floors = num_floors
        self.q_table = {}

    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, state):
        # Epsilon-greedy policy
        if random.random() < self.epsilon:
            return random.choice(ACTIONS)
        else:
            q_values = [self.get_q(state, a) for a in ACTIONS]
            max_q = max(q_values)
            max_actions = [a for a, q in zip(ACTIONS, q_values) if q == max_q]
            return random.choice(max_actions)

    def learn(self, state, action, reward, next_state, done):
        current_q = self.get_q(state, action)
        if done:
            target = reward
        else:
            next_qs = [self.get_q(next_state, a) for a in ACTIONS]
            target = reward + self.gamma * max(next_qs)

        new_q = current_q + self.alpha * (target - current_q)
        self.q_table[(state, action)] = new_q

        # Decay epsilon after each episode
        if done and self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

print("✓ Environment and Agent classes defined")

✓ Environment and Agent classes defined


#Visualization and Simulation Class

In [4]:
# =========================================
# Define the pygame-based visualization and main simulation logic

class ElevatorSim:
    """
    Simulator combining env, agent, visualization, fast training, and model saving.
    """

    def __init__(self):
        pygame.init()
        pygame.display.set_caption("Elevator RL Simulation with Fast Training & Model Save")
        self.screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont(None, 22)
        self.big_font = pygame.font.SysFont(None, 30, bold=True)

        self.env = ElevatorEnv()
        self.agent = QLearningAgent(FLOORS)
        self.load_model_if_exists()

        self.state = self.env.reset()
        self.total_rewards = 0
        self.best_reward = float('-inf')
        self.episode = 1
        self.training = True

        self.elevator_y_pos = SCREEN_HEIGHT - (self.env.current_floor + 1)*FLOOR_HEIGHT + 5
        self.move_speed = ELEVATOR_SPEED

        self.door_open_frames = 0
        self.DOOR_OPEN_DURATION = 10  # shorter door open for faster training

        self.max_training_episodes = 500  # Reduced to 500 episodes as requested
        self.evaluation_episodes = 50
        self.eval_episode_counter = 0

    def load_model_if_exists(self):
        if os.path.exists(MODEL_SAVE_PATH):
            with open(MODEL_SAVE_PATH, 'rb') as f:
                self.agent.q_table = pickle.load(f)
            print("Loaded existing trained Q-table from disk.")
            self.agent.epsilon = self.agent.epsilon_min  # disable exploration for loaded model

    def save_model(self):
        with open(MODEL_SAVE_PATH, 'wb') as f:
            pickle.dump(self.agent.q_table, f)
        print(f"Model saved to {MODEL_SAVE_PATH}")

    def draw_background(self):
        self.screen.fill(WHITE)
        for i in range(FLOORS):
            y = SCREEN_HEIGHT - (i + 1)*FLOOR_HEIGHT
            pygame.draw.line(self.screen, GRAY, (0, y), (SCREEN_WIDTH, y), 3)
            floor_label = FLOOR_NAMES[i]
            text = self.big_font.render(f"Floor {floor_label}", True, BLACK)
            self.screen.blit(text, (10, y + FLOOR_HEIGHT//2 - text.get_height()//2))

    def draw_requests(self):
        for i, dest_list in enumerate(self.env.pickup_requests):
            count = len(dest_list)
            if count > 0:
                y = SCREEN_HEIGHT - (i + 1)*FLOOR_HEIGHT + FLOOR_HEIGHT//2
                x = SCREEN_WIDTH - 80
                pygame.draw.circle(self.screen, RED, (x, y), 15)
                text = self.font.render(f"{count}", True, WHITE)
                self.screen.blit(text, (x - text.get_width()//2, y - text.get_height()//2))

    def draw_passengers_inside(self):
        count = len(self.env.passengers)
        if count > 0:
            x = 100 + ELEVATOR_WIDTH//2
            y = int(self.elevator_y_pos) + ELEVATOR_HEIGHT//2
            pygame.draw.circle(self.screen, GREEN, (x, y), 18)
            text = self.big_font.render(f"{count}", True, WHITE)
            self.screen.blit(text, (x - text.get_width()//2, y - text.get_height()//2))

    def draw_elevator(self):
        x = 100
        y = int(self.elevator_y_pos)
        pygame.draw.rect(self.screen, BLUE, (x, y, ELEVATOR_WIDTH, ELEVATOR_HEIGHT), border_radius=5)

        door_color = YELLOW if self.door_open_frames > 0 else BLACK
        door_width = ELEVATOR_WIDTH // 4
        door_height = ELEVATOR_HEIGHT - 20

        left_door_x = x + 10
        left_door_y = y + 10
        right_door_x = x + ELEVATOR_WIDTH - door_width - 10
        right_door_y = y + 10

        if self.door_open_frames > 0:
            gap = door_width // 2
            pygame.draw.rect(self.screen, door_color, (left_door_x - gap, left_door_y, door_width, door_height), border_radius=3)
            pygame.draw.rect(self.screen, door_color, (right_door_x + gap, right_door_y, door_width, door_height), border_radius=3)
            self.door_open_frames -= 1
        else:
            pygame.draw.rect(self.screen, door_color, (left_door_x, left_door_y, door_width, door_height), border_radius=3)
            pygame.draw.rect(self.screen, door_color, (right_door_x, right_door_y, door_width, door_height), border_radius=3)

        floor_idx = self.env.current_floor
        floor_label = FLOOR_NAMES[floor_idx]
        floor_text = self.big_font.render(f"Floor {floor_label}", True, WHITE)
        self.screen.blit(floor_text, (x + 10, y + ELEVATOR_HEIGHT + 5))

        self.draw_passengers_inside()

    def draw_info(self):
        info_texts = [
            f"Episode: {self.episode}",
            f"Total Reward: {self.total_rewards:.1f}",
            f"Epsilon: {self.agent.epsilon:.3f}",
            f"Pending Pickups: {self.env.pending_pickups_count()}",
            f"Passengers Inside: {self.env.passengers_count()}",
        ]

        for i, text in enumerate(info_texts):
            rendered = self.font.render(text, True, BLACK)
            self.screen.blit(rendered, (10, 10 + i * 20))

    def update_elevator_position(self):
        target_y = SCREEN_HEIGHT - (self.env.current_floor + 1)*FLOOR_HEIGHT + 5
        if abs(self.elevator_y_pos - target_y) > self.move_speed:
            if self.elevator_y_pos < target_y:
                self.elevator_y_pos += self.move_speed
            else:
                self.elevator_y_pos -= self.move_speed
        else:
            self.elevator_y_pos = target_y

print("✓ Visualization and Simulation class defined")

✓ Visualization and Simulation class defined


#Main Execution and Training Loop

In [5]:
# =======================================
# Execute the simulation with training and evaluation phases

def run_simulation():
    """
    Main function to run the elevator RL simulation
    """
    sim = ElevatorSim()

    # Main simulation loop with training and evaluation
    def simulation_loop():
        running = True

        while running:
            sim.clock.tick(FPS)

            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

            if sim.training:
                action = sim.agent.choose_action(sim.state)
                next_state, reward, done, _ = sim.env.step(action)
                sim.agent.learn(sim.state, action, reward, next_state, done)
                sim.state = next_state
                sim.total_rewards += reward
                sim.update_elevator_position()

                if action == ACTION_STAY:
                    sim.door_open_frames = sim.DOOR_OPEN_DURATION

                if done:
                    # Update best model and save
                    if sim.total_rewards > sim.best_reward:
                        sim.best_reward = sim.total_rewards
                        sim.save_model()
                    print(f"Episode {sim.episode} done. Total reward: {sim.total_rewards:.1f}. Best reward: {sim.best_reward:.1f}")
                    sim.episode += 1
                    sim.state = sim.env.reset()
                    sim.total_rewards = 0
                    sim.elevator_y_pos = SCREEN_HEIGHT - (sim.env.current_floor + 1)*FLOOR_HEIGHT + 5
                    # Decay epsilon
                    sim.agent.epsilon = max(sim.agent.epsilon * sim.agent.epsilon_decay, sim.agent.epsilon_min)
                    # Stop training after max episodes
                    if sim.episode > sim.max_training_episodes:
                        print("Training completed. Switching to evaluation mode.")
                        sim.training = False
                        sim.eval_episode_counter = 0
                        sim.agent.epsilon = 0.0  # No exploration in evaluation
                        sim.state = sim.env.reset()
                        sim.total_rewards = 0

            else:
                # Evaluation mode - no exploration
                action = sim.agent.choose_action(sim.state)
                next_state, reward, done, _ = sim.env.step(action)
                sim.state = next_state
                sim.total_rewards += reward
                sim.update_elevator_position()

                if action == ACTION_STAY:
                    sim.door_open_frames = sim.DOOR_OPEN_DURATION

                if done:
                    sim.eval_episode_counter += 1
                    print(f"Evaluation episode {sim.eval_episode_counter} done with total reward: {sim.total_rewards:.1f}")
                    sim.total_rewards = 0
                    sim.state = sim.env.reset()
                    sim.elevator_y_pos = SCREEN_HEIGHT - (sim.env.current_floor + 1)*FLOOR_HEIGHT + 5
                    if sim.eval_episode_counter >= sim.evaluation_episodes:
                        print("Evaluation complete. Exiting...")
                        running = False

            # Drawing
            sim.draw_background()
            sim.draw_requests()
            sim.draw_elevator()
            sim.draw_info()

            pygame.display.flip()

        pygame.quit()
        return sim

    return simulation_loop()

# Run the simulation
print("🚀 Starting Elevator RL Simulation...")
print("📊 Training for 500 episodes, then evaluating for 50 episodes")
print("💾 Best models will be automatically saved")
print("🎮 Close the pygame window to stop the simulation")

# Uncomment the line below to run the simulation
run_simulation()

print("✓ Simulation setup complete. Uncomment the last line to run!")

🚀 Starting Elevator RL Simulation...
📊 Training for 500 episodes, then evaluating for 50 episodes
💾 Best models will be automatically saved
🎮 Close the pygame window to stop the simulation
Model saved to best_q_table.pkl
Episode 1 done. Total reward: 43.0. Best reward: 43.0
Model saved to best_q_table.pkl
Episode 2 done. Total reward: 1481.0. Best reward: 1481.0
Model saved to best_q_table.pkl
Episode 3 done. Total reward: 2262.0. Best reward: 2262.0
Model saved to best_q_table.pkl
Episode 4 done. Total reward: 2897.0. Best reward: 2897.0
Episode 5 done. Total reward: 2180.0. Best reward: 2897.0
Episode 6 done. Total reward: 1171.0. Best reward: 2897.0
Episode 7 done. Total reward: 203.0. Best reward: 2897.0
Episode 8 done. Total reward: 40.0. Best reward: 2897.0
Episode 9 done. Total reward: 2598.0. Best reward: 2897.0
Episode 10 done. Total reward: 1464.0. Best reward: 2897.0
Episode 11 done. Total reward: 1478.0. Best reward: 2897.0
Episode 12 done. Total reward: 2051.0. Best reward

#comparison between trained and random agent

#Imports and Configuration

In [6]:
# ================================
# Import libraries and define constants for side-by-side elevator comparison

import pygame
import random
import numpy as np
import sys
import pickle
import os
import time

# ================================
# Constants for Side-by-Side Simulation
# ================================
FPS = 120  # High FPS for smooth fast simulation
SCREEN_WIDTH = 900  # Width doubled for side-by-side view
SCREEN_HEIGHT = 600

FLOORS = 6  # Basement + 5 floors above
FLOOR_HEIGHT = SCREEN_HEIGHT // FLOORS
ELEVATOR_WIDTH = 80
ELEVATOR_HEIGHT = FLOOR_HEIGHT - 10
ELEVATOR_SPEED = 30  # pixels per frame

# ================================
# Reinforcement Learning Parameters
# ================================
ALPHA = 0.1
GAMMA = 0.95
EPSILON_START = 0.0  # No exploration in evaluation mode
EPSILON_END = 0.0
EPSILON_DECAY = 1.0

# ================================
# Action Definitions
# ================================
ACTION_UP = 0
ACTION_DOWN = 1
ACTION_STAY = 2
ACTIONS = [ACTION_UP, ACTION_DOWN, ACTION_STAY]

# ================================
# UI Colors
# ================================
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GRAY = (230, 230, 230)
RED = (220, 50, 50)
GREEN = (50, 180, 50)
BLUE = (50, 50, 200)
YELLOW = (240, 230, 140)

# ================================
# Floor Names and Model Paths
# ================================
FLOOR_NAMES = ['B', '1', '2', '3', '4', '5']
MODEL1_PATH = "best_q_table.pkl"
MODEL2_PATH = "random_q_table.pkl"  # For demonstration

# ================================
# Simulation Settings
# ================================
SIMULATION_DURATION = 120  # Duration in seconds for comparison

print("✓ Libraries imported and side-by-side comparison configuration set")

✓ Libraries imported and side-by-side comparison configuration set


#Environment and Agent Classes

In [7]:
# ====================================
# Define elevator environment and comparison agents (trained vs random)

class ElevatorEnv:
    """
    Elevator environment for side-by-side comparison.
    Extended with longer max_time_steps for 2-minute comparison runs.
    """

    def __init__(self, max_time_steps=1800):  # Large max steps to cover ~2 minutes with FPS=120
        self.num_floors = FLOORS
        self.current_floor = 0
        self.pickup_requests = [[] for _ in range(self.num_floors)]
        self.passengers = []
        self.time_step = 0
        self.max_time_steps = max_time_steps

    def reset(self):
        self.current_floor = 0
        self.pickup_requests = [[] for _ in range(self.num_floors)]
        self.passengers = []
        self.time_step = 0

        # Initial random requests for simulation start
        for _ in range(5):
            origin = random.randint(0, self.num_floors - 1)
            dest = random.randint(0, self.num_floors - 1)
            while dest == origin:
                dest = random.randint(0, self.num_floors - 1)
            self.pickup_requests[origin].append(dest)

        return self._get_state()

    def _get_state(self):
        """State representation for Q-learning agent."""
        pickup_vec = tuple(int(len(self.pickup_requests[floor]) > 0) for floor in range(self.num_floors))
        drop_vec = [0]*self.num_floors
        for dest_floor in self.passengers:
            drop_vec[dest_floor] = 1
        drop_vec = tuple(drop_vec)
        return (self.current_floor, pickup_vec, drop_vec)

    def step(self, action):
        """Execute action and return new state, reward, done flag."""
        self.time_step += 1
        waiting_passengers_count = sum(len(dest_list) for dest_list in self.pickup_requests)
        negative_reward = -1 * waiting_passengers_count
        reward = negative_reward
        done = False

        # Generate new passengers probabilistically
        self._generate_new_passengers()

        # Execute action
        if action == ACTION_UP and self.current_floor < self.num_floors - 1:
            self.current_floor += 1
        elif action == ACTION_DOWN and self.current_floor > 0:
            self.current_floor -= 1
        elif action == ACTION_STAY:
            # Drop off passengers
            drop_count = self.passengers.count(self.current_floor)
            reward += drop_count * 30
            self.passengers = [d for d in self.passengers if d != self.current_floor]

            # Pick up waiting passengers
            pickup_count = len(self.pickup_requests[self.current_floor])
            reward += pickup_count * 15
            self.passengers.extend(self.pickup_requests[self.current_floor])
            self.pickup_requests[self.current_floor] = []

        if self.time_step >= self.max_time_steps:
            done = True

        new_state = self._get_state()
        return new_state, reward, done, {}

    def _generate_new_passengers(self):
        """Generate new passenger requests randomly."""
        prob_new_passenger = 0.1
        for floor in range(self.num_floors):
            if random.random() < prob_new_passenger:
                origin = floor
                dest = random.randint(0, self.num_floors - 1)
                while dest == origin:
                    dest = random.randint(0, self.num_floors - 1)
                self.pickup_requests[origin].append(dest)

    def pending_pickups_count(self):
        return sum(len(dest_list) for dest_list in self.pickup_requests)

    def passengers_count(self):
        return len(self.passengers)


class QLearningAgent:
    """
    Trained Q-Learning agent loaded from saved model.
    Uses greedy policy (no exploration) for evaluation.
    """

    def __init__(self, num_floors, q_table=None):
        self.alpha = ALPHA
        self.gamma = GAMMA
        self.epsilon = EPSILON_START  # 0.0 for evaluation
        self.epsilon_min = EPSILON_END
        self.epsilon_decay = EPSILON_DECAY
        self.num_floors = num_floors
        self.q_table = q_table if q_table else {}

    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, state):
        """Greedy policy - choose action with highest Q-value."""
        q_values = [self.get_q(state, a) for a in ACTIONS]
        max_q = max(q_values)
        max_actions = [a for a, q in zip(ACTIONS, q_values) if q == max_q]
        return random.choice(max_actions)

    def learn(self, state, action, reward, next_state, done):
        """No learning during evaluation phase."""
        pass


class RandomAgent:
    """
    Random baseline agent for comparison.
    Chooses actions completely randomly.
    """

    def __init__(self, num_floors):
        self.num_floors = num_floors

    def choose_action(self, state):
        """Choose random action."""
        return random.choice(ACTIONS)

    def learn(self, *args, **kwargs):
        """No learning for random agent."""
        pass

print("✓ Environment and Agent classes (Trained Q-Learning vs Random) defined")

✓ Environment and Agent classes (Trained Q-Learning vs Random) defined


#Side-by-Side Visualization Class

In [8]:
# =======================================
# Define the pygame-based side-by-side visualization system

class ElevatorSimSideBySide:
    """
    Side-by-side elevator simulation comparing two algorithms:
    - Left side: Trained Q-Learning Agent
    - Right side: Random Agent
    """

    def __init__(self):
        pygame.init()
        pygame.display.set_caption("Elevator RL Algorithms Comparison")
        self.screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont(None, 20)
        self.big_font = pygame.font.SysFont(None, 28, bold=True)

        # Initialize both environments (identical starting conditions)
        self.env1 = ElevatorEnv()
        self.env2 = ElevatorEnv()

        # Load trained Q-Learning agent (Algorithm 1)
        q_table1 = {}
        if os.path.exists(MODEL1_PATH):
            with open(MODEL1_PATH, "rb") as f:
                q_table1 = pickle.load(f)
            print(f"✓ Loaded trained Q-table from {MODEL1_PATH}")
        else:
            print(f"⚠️  No trained model found at {MODEL1_PATH}, using empty Q-table")
        self.agent1 = QLearningAgent(FLOORS, q_table=q_table1)

        # Random agent (Algorithm 2)
        self.agent2 = RandomAgent(FLOORS)

        # Initialize states
        self.state1 = self.env1.reset()
        self.state2 = self.env2.reset()

        # Elevator positions for animation
        self.elevator_y_pos1 = SCREEN_HEIGHT - (self.env1.current_floor + 1) * FLOOR_HEIGHT + 5
        self.elevator_y_pos2 = SCREEN_HEIGHT - (self.env2.current_floor + 1) * FLOOR_HEIGHT + 5

        # Door animation frames
        self.door_open_frames1 = 0
        self.door_open_frames2 = 0
        self.DOOR_OPEN_DURATION = 10

        # Performance tracking
        self.total_rewards1 = 0
        self.total_rewards2 = 0

        # Timing for comparison
        self.start_time = time.time()

    def draw_background(self):
        """Draw the split-screen background with floors."""
        self.screen.fill(WHITE)

        # Center dividing line
        center_line_x = SCREEN_WIDTH // 2
        pygame.draw.line(self.screen, BLACK, (center_line_x, 0), (center_line_x, SCREEN_HEIGHT), 3)

        # Draw floor lines and labels for both sides
        for i in range(FLOORS):
            y = SCREEN_HEIGHT - (i + 1) * FLOOR_HEIGHT
            floor_label = FLOOR_NAMES[i]

            # Left side floor
            pygame.draw.line(self.screen, GRAY, (0, y), (center_line_x, y), 2)
            text = self.big_font.render(f"Floor {floor_label}", True, BLACK)
            self.screen.blit(text, (10, y + FLOOR_HEIGHT//2 - text.get_height()//2))

            # Right side floor
            pygame.draw.line(self.screen, GRAY, (center_line_x, y), (SCREEN_WIDTH, y), 2)
            self.screen.blit(text, (center_line_x + 10, y + FLOOR_HEIGHT//2 - text.get_height()//2))

    def draw_requests(self, env, offset_x):
        """Draw pickup requests as red circles with passenger counts."""
        for i, dest_list in enumerate(env.pickup_requests):
            count = len(dest_list)
            if count > 0:
                y = SCREEN_HEIGHT - (i + 1)*FLOOR_HEIGHT + FLOOR_HEIGHT//2
                x = offset_x + 100
                pygame.draw.circle(self.screen, RED, (x, y), 15)
                text = self.font.render(f"{count}", True, WHITE)
                self.screen.blit(text, (x - text.get_width()//2, y - text.get_height()//2))

    def draw_passengers_inside(self, env, elevator_x, elevator_y_pos):
        """Draw passengers inside elevator as green circle with count."""
        count = len(env.passengers)
        if count > 0:
            x = elevator_x + ELEVATOR_WIDTH//2
            y = int(elevator_y_pos) + ELEVATOR_HEIGHT//2
            pygame.draw.circle(self.screen, GREEN, (x, y), 18)
            text = self.big_font.render(f"{count}", True, WHITE)
            self.screen.blit(text, (x - text.get_width()//2, y - text.get_height()//2))

    def draw_elevator(self, env, elevator_y_pos, offset_x, door_open_frames, label):
        """Draw elevator with doors, passengers, and algorithm label."""
        x = offset_x + 100
        y = int(elevator_y_pos)

        # Elevator body
        pygame.draw.rect(self.screen, BLUE, (x, y, ELEVATOR_WIDTH, ELEVATOR_HEIGHT), border_radius=5)

        # Door animation
        door_color = YELLOW if door_open_frames > 0 else BLACK
        door_width = ELEVATOR_WIDTH // 4
        door_height = ELEVATOR_HEIGHT - 20
        left_door_x = x + 10
        left_door_y = y + 10
        right_door_x = x + ELEVATOR_WIDTH - door_width - 10
        right_door_y = y + 10

        if door_open_frames > 0:
            gap = door_width // 2
            pygame.draw.rect(self.screen, door_color, (left_door_x - gap, left_door_y, door_width, door_height), border_radius=3)
            pygame.draw.rect(self.screen, door_color, (right_door_x + gap, right_door_y, door_width, door_height), border_radius=3)
        else:
            pygame.draw.rect(self.screen, door_color, (left_door_x, left_door_y, door_width, door_height), border_radius=3)
            pygame.draw.rect(self.screen, door_color, (right_door_x, right_door_y, door_width, door_height), border_radius=3)

        # Floor indicator
        floor_idx = env.current_floor
        floor_label = FLOOR_NAMES[floor_idx]
        floor_text = self.big_font.render(f"Floor {floor_label}", True, WHITE)
        self.screen.blit(floor_text, (x + 10, y + ELEVATOR_HEIGHT + 5))

        # Draw passengers inside
        self.draw_passengers_inside(env, x, elevator_y_pos)

        # Algorithm label above elevator
        label_text = self.big_font.render(label, True, BLACK)
        self.screen.blit(label_text, (x - 20, y - 40))

    def draw_info(self):
        """Draw performance information and timer."""
        left_x = 0
        right_x = SCREEN_WIDTH // 2

        # Left side info (Trained Agent)
        info1 = [
            f"Total Reward: {self.total_rewards1:.1f}",
            f"Pending Pickups: {self.env1.pending_pickups_count()}",
            f"Passengers Inside: {self.env1.passengers_count()}",
        ]

        # Right side info (Random Agent)
        info2 = [
            f"Total Reward: {self.total_rewards2:.1f}",
            f"Pending Pickups: {self.env2.pending_pickups_count()}",
            f"Passengers Inside: {self.env2.passengers_count()}",
        ]

        # Draw performance stats
        for i, text in enumerate(info1):
            rendered = self.font.render(text, True, BLACK)
            self.screen.blit(rendered, (left_x + 10, SCREEN_HEIGHT - (len(info1) - i) * 22))

        for i, text in enumerate(info2):
            rendered = self.font.render(text, True, BLACK)
            self.screen.blit(rendered, (right_x + 10, SCREEN_HEIGHT - (len(info2) - i) * 22))

        # Timer display
        elapsed = int(time.time() - self.start_time)
        timer_text = self.font.render(f"Time Elapsed: {elapsed} / {SIMULATION_DURATION} sec", True, BLACK)
        self.screen.blit(timer_text, (SCREEN_WIDTH//2 - timer_text.get_width()//2, 5))

    def update_elevator_position(self, y_pos, current_floor):
        """Smoothly animate elevator movement between floors."""
        target_y = SCREEN_HEIGHT - (current_floor + 1) * FLOOR_HEIGHT + 5
        if abs(y_pos - target_y) > ELEVATOR_SPEED:
            if y_pos < target_y:
                y_pos += ELEVATOR_SPEED
            else:
                y_pos -= ELEVATOR_SPEED
        else:
            y_pos = target_y
        return y_pos

print("✓ Side-by-side visualization class defined")

✓ Side-by-side visualization class defined


#Main Execution and Comparison Loop

In [11]:
# =========================================
# Execute the side-by-side algorithm comparison with real-time performance tracking

def run_comparison_simulation():
    """
    Main function to run the side-by-side elevator algorithm comparison.
    Compares trained Q-Learning agent vs Random agent over 2 minutes.
    """

    # Add the main simulation loop to the ElevatorSimSideBySide class
    def simulation_run_method(self):
        """Main simulation loop comparing two algorithms side-by-side."""
        running = True

        print("🚀 Starting side-by-side algorithm comparison...")
        print("📊 Left: Trained Q-Learning Agent | Right: Random Agent")
        print(f"⏱️  Running for {SIMULATION_DURATION} seconds")

        while running:
            self.clock.tick(FPS)

            # Handle pygame events
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

            # ========== AGENT 1 (Trained Q-Learning) STEP ==========
            action1 = self.agent1.choose_action(self.state1)
            next_state1, reward1, done1, _ = self.env1.step(action1)
            self.state1 = next_state1
            self.total_rewards1 += reward1

            # Handle door animation for agent 1
            if action1 == ACTION_STAY:
                self.door_open_frames1 = self.DOOR_OPEN_DURATION
            if self.door_open_frames1 > 0:
                self.door_open_frames1 -= 1

            # Update elevator position for agent 1
            self.elevator_y_pos1 = self.update_elevator_position(
                self.elevator_y_pos1, self.env1.current_floor
            )

            # ========== AGENT 2 (Random) STEP ==========
            action2 = self.agent2.choose_action(self.state2)
            next_state2, reward2, done2, _ = self.env2.step(action2)
            self.state2 = next_state2
            self.total_rewards2 += reward2

            # Handle door animation for agent 2
            if action2 == ACTION_STAY:
                self.door_open_frames2 = self.DOOR_OPEN_DURATION
            if self.door_open_frames2 > 0:
                self.door_open_frames2 -= 1

            # Update elevator position for agent 2
            self.elevator_y_pos2 = self.update_elevator_position(
                self.elevator_y_pos2, self.env2.current_floor
            )

            # ========== ENVIRONMENT RESET IF NEEDED ==========
            if done1:
                self.state1 = self.env1.reset()
                self.elevator_y_pos1 = SCREEN_HEIGHT - (self.env1.current_floor + 1) * FLOOR_HEIGHT + 5
                self.door_open_frames1 = 0
                print("📈 Environment 1 (Trained Agent) reset")

            if done2:
                self.state2 = self.env2.reset()
                self.elevator_y_pos2 = SCREEN_HEIGHT - (self.env2.current_floor + 1) * FLOOR_HEIGHT + 5
                self.door_open_frames2 = 0
                print("📈 Environment 2 (Random Agent) reset")

            # ========== DRAWING AND VISUALIZATION ==========
            self.draw_background()

            # Left side - Trained Agent
            self.draw_requests(self.env1, 0)
            self.draw_elevator(self.env1, self.elevator_y_pos1, 0,
                             self.door_open_frames1, "Trained Agent")

            # Right side - Random Agent
            self.draw_requests(self.env2, SCREEN_WIDTH // 2)
            self.draw_elevator(self.env2, self.elevator_y_pos2, SCREEN_WIDTH // 2,
                             self.door_open_frames2, "Random Agent")

            # Draw performance information
            self.draw_info()
            pygame.display.flip()

            # ========== TIME-BASED TERMINATION ==========
            elapsed = time.time() - self.start_time
            if elapsed >= SIMULATION_DURATION:
                print("\n" + "="*60)
                print("🏁 SIMULATION COMPLETE - FINAL RESULTS")
                print("="*60)
                print(f"⏱️  Total Runtime: {elapsed:.1f} seconds")
                print(f"🤖 Trained Q-Learning Agent:")
                print(f"   └── Total Reward: {self.total_rewards1:.1f}")
                print(f"   └── Final Pending Pickups: {self.env1.pending_pickups_count()}")
                print(f"   └── Final Passengers Inside: {self.env1.passengers_count()}")
                print(f"🎲 Random Agent:")
                print(f"   └── Total Reward: {self.total_rewards2:.1f}")
                print(f"   └── Final Pending Pickups: {self.env2.pending_pickups_count()}")
                print(f"   └── Final Passengers Inside: {self.env2.passengers_count()}")

                # Performance comparison
                performance_diff = self.total_rewards1 - self.total_rewards2
                print(f"\n📊 PERFORMANCE COMPARISON:")
                print(f"   └── Reward Difference: {performance_diff:.1f}")
                if performance_diff > 0:
                    print(f"   └── 🏆 Trained Agent outperformed Random Agent!")
                elif performance_diff < 0:
                    print(f"   └── 😮 Random Agent outperformed Trained Agent!")
                else:
                    print(f"   └── 🤝 Both agents performed equally!")
                print("="*60)

                running = False

        pygame.quit()
        return {
            'trained_reward': self.total_rewards1,
            'random_reward': self.total_rewards2,
            'performance_difference': self.total_rewards1 - self.total_rewards2,
            'duration': elapsed
        }

    # Monkey patch the run method to the class
    ElevatorSimSideBySide.run = simulation_run_method

    # Create and run simulation
    sim = ElevatorSimSideBySide()
    results = sim.run()
    return results

# Function to run the comparison
def start_elevator_comparison():
    """
    Convenience function to start the elevator algorithm comparison.
    Returns performance results for analysis.
    """
    print("🎮 Starting Elevator RL Algorithm Comparison")
    print("📋 Make sure you have 'best_q_table.pkl' in your current directory")
    print("🖼️  Close the pygame window or wait 2 minutes to end simulation")
    print("-" * 50)

    try:
        results = run_comparison_simulation()
        return results
    except Exception as e:
        print(f"❌ Error during simulation: {e}")
        return None

print("✓ Side-by-side comparison execution system ready")
print("\n🎯 To run the comparison, execute:")
print("   results = start_elevator_comparison()")
print("\n📁 Make sure 'best_q_table.pkl' exists in your directory for the trained agent")

# Uncomment the line below to run the comparison
results = start_elevator_comparison()

✓ Side-by-side comparison execution system ready

🎯 To run the comparison, execute:
   results = start_elevator_comparison()

📁 Make sure 'best_q_table.pkl' exists in your directory for the trained agent
🎮 Starting Elevator RL Algorithm Comparison
📋 Make sure you have 'best_q_table.pkl' in your current directory
🖼️  Close the pygame window or wait 2 minutes to end simulation
--------------------------------------------------
✓ Loaded trained Q-table from best_q_table.pkl
🚀 Starting side-by-side algorithm comparison...
📊 Left: Trained Q-Learning Agent | Right: Random Agent
⏱️  Running for 120 seconds
📈 Environment 1 (Trained Agent) reset
📈 Environment 2 (Random Agent) reset
📈 Environment 1 (Trained Agent) reset
📈 Environment 2 (Random Agent) reset
📈 Environment 1 (Trained Agent) reset
📈 Environment 2 (Random Agent) reset
📈 Environment 1 (Trained Agent) reset
📈 Environment 2 (Random Agent) reset
📈 Environment 1 (Trained Agent) reset
📈 Environment 2 (Random Agent) reset
📈 Environment 1 (T